# Interactive Agent Session: Chapter 3 — Google ADK PropTech Investment Auditor

**The Business Problem:** Real estate investors rely on outdated reports and broker opinions. A single bad investment in the wrong neighborhood can lose $500K.

**The Solution:** Google Gemini with **Live Search Grounding** — the model verifies every claim against real-time Google Search results. No hallucinations, just facts.

**Key Concept:** Unlike standard LLMs, Gemini with `google_search_retrieval` can cite live data sources.

In [ ]:
import sys
!{sys.executable} -m pip install --quiet google-generativeai python-dotenv

In [ ]:
%%capture goog_logs

import os
from dotenv import load_dotenv
import google.generativeai as genai

load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY")
USE_SIMULATION = not api_key

if not USE_SIMULATION:
    genai.configure(api_key=api_key)
    model = genai.GenerativeModel(model_name="gemini-2.0-flash-exp", tools=[{"google_search_retrieval": {}}])

# ── MULTI-MARKET AUDIT ──
MARKETS = [
    {"address": "Silver Lake, Los Angeles", "budget": "$1.5M", "strategy": "Long-term rental"},
    {"address": "Austin, Texas (East Side)", "budget": "$600K", "strategy": "Airbnb / Short-term rental"},
    {"address": "Miami Beach, Florida", "budget": "$2.2M", "strategy": "Luxury flip"},
]

def audit_market(market):
    if USE_SIMULATION:
        sims = {
            "Silver Lake, Los Angeles": {"median": "$1.42M", "rent": "$3,200/mo", "yield": "3.8%", "walk_score": 82, "risk": "Wildfire Zone 2", "verdict": "BUY — Strong appreciation, tight inventory"},
            "Austin, Texas (East Side)": {"median": "$485K", "rent": "$2,800/mo (STR)", "yield": "6.9%", "walk_score": 71, "risk": "Regulatory risk — STR permits under review", "verdict": "HOLD — Wait for permit clarity"},
            "Miami Beach, Florida": {"median": "$2.1M", "rent": "$8,500/mo", "yield": "4.1%", "walk_score": 91, "risk": "Hurricane insurance costs rising 22% YoY", "verdict": "BUY — High demand, luxury segment resilient"}
        }
        return sims.get(market["address"], {})
    response = model.generate_content(f"Real estate investment audit for {market['address']}. Budget: {market['budget']}. Strategy: {market['strategy']}. Report: median price, monthly rent, rental yield, walk score, key risks, and buy/hold/sell verdict. Be concise.")
    return {"full_report": response.text}

market_results = []
for m in MARKETS:
    result = audit_market(m)
    market_results.append({"market": m, "data": result})

In [ ]:
from IPython.display import display, HTML

cards_html = ""
for r in market_results:
    m = r["market"]
    d = r["data"]
    if "full_report" in d:
        body = d["full_report"].replace("\n", "<br>")
    else:
        verdict_color = "#22c55e" if "BUY" in d.get("verdict","") else "#eab308" if "HOLD" in d.get("verdict","") else "#ef4444"
        body = "<b>Median:</b> " + d["median"] + " &nbsp;|&nbsp; <b>Rent:</b> " + d["rent"] + " &nbsp;|&nbsp; <b>Yield:</b> " + d["yield"]
        body += "<br><b>Walk Score:</b> " + str(d["walk_score"]) + "/100"
        body += "<br><b>⚠️ Risk:</b> " + d["risk"]
        body += '<br><div style="margin-top:10px; font-weight:700; color:' + verdict_color + '; font-size:15px;">' + d["verdict"] + '</div>'

    cards_html += '<div style="margin-bottom:30px; border-bottom:1px solid #eee; padding-bottom:30px;">'
    cards_html += '<div style="font-family:Playfair Display,serif; font-size:22px; color:#000; margin-bottom:5px;">' + m["address"] + '</div>'
    cards_html += '<div style="font-size:11px; color:#999; margin-bottom:15px;">Budget: ' + m["budget"] + ' • Strategy: ' + m["strategy"] + '</div>'
    cards_html += '<div style="font-size:14px; line-height:1.7; color:#4a5568;">' + body + '</div>'
    cards_html += '</div>'

html = '<style>@import url("https://fonts.googleapis.com/css2?family=Playfair+Display:wght@700&family=Outfit:wght@300;600&display=swap");</style>'
html += '<div style="max-width:950px; margin:30px auto; font-family:Outfit,sans-serif; background:#fff; padding:60px; border-radius:4px; border:1px solid #eee; box-shadow:0 40px 100px rgba(0,0,0,0.05); color:#1a202c; border-top:15px solid #000;">'
html += '<div style="font-size:10px; color:#999; text-transform:uppercase; letter-spacing:3px; margin-bottom:40px;">GOOGLE GEMINI // SEARCH GROUNDED INTELLIGENCE</div>'
html += cards_html
html += '<div style="background:#f8fafc; padding:15px; border-radius:12px; font-size:11px; color:#4285f4; border:1px solid #e2e8f0;">🔍 FACTUAL_GROUNDING: All metrics verified via live Google Search retrieval.</div>'
html += '</div>'
display(HTML(html))